# Leao Claro: Assistente RAG para Duvidas de IRPF

## Assistente de orientacao tributaria baseado em documentos oficiais

### Objetivo
Construir um assistente conversacional com **Retrieval-Augmented Generation (RAG)** para responder duvidas sobre **Imposto de Renda Pessoa Fisica**, usando como base o documento oficial de Perguntas e Respostas IRPF 2026 da Receita Federal.

---

### O que este projeto demonstra
- Carregamento de documentos oficiais em PDF
- Divisao do documento em blocos de texto
- Criacao de embeddings com Gemini
- Armazenamento vetorial com ChromaDB
- Recuperacao de contexto relevante
- Respostas com base documental e trechos de apoio
- Cuidados de escopo para um tema sensivel

> Este projeto e educacional e informativo. Ele nao substitui contador, advogado, consultor tributario nem os canais oficiais da Receita Federal.

## Dependencias

Execute no terminal antes de rodar o notebook:

```bash
pip install -U langchain langchain-community langchain-text-splitters langchain-google-genai langchain-chroma chromadb pypdf python-dotenv
```

Para rodar localmente, crie uma chave gratuita no Google AI Studio e salve como `GOOGLE_API_KEY` no arquivo `.env`.

Google AI Studio: https://aistudio.google.com/app/apikey

Para publicar uma demo publica, nao inclua `.env` no repositorio. O app Gradio desta pasta foi pensado para pedir a chave do proprio usuario.

In [ ]:

# Importações básicas
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

# Loader de documentos PDF
from langchain_community.document_loaders import PyPDFLoader

# Divisão de texto em blocos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Banco vetorial
from langchain_chroma import Chroma

# LLM
from langchain_google_genai import ChatGoogleGenerativeAI

# Prompt RAG
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

for parent in [Path.cwd(), *Path.cwd().parents]:
    if parent.name == "PROJETO EMPREGO":
        load_dotenv(parent / "Time de Agents" / ".env")
        break

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass("Cole sua GOOGLE_API_KEY do Google AI Studio: ")

GEMINI_LLM_MODEL = "gemini-2.5-flash-lite"
GEMINI_EMBEDDING_MODEL = "models/gemini-embedding-001"


## 1. Definicao do Problema

Declarar o Imposto de Renda pode ser confuso porque as regras envolvem muitos casos especificos, documentos longos e termos tecnicos.

LLMs tambem podem alucinar quando respondem sem uma fonte confiavel. Por isso, este projeto usa RAG: o modelo so responde depois de recuperar trechos relevantes do documento oficial da Receita Federal.

A proposta do Leao Claro e ajudar o usuario a entender melhor o material oficial, sem coletar dados sensiveis e sem substituir profissionais especializados.

In [ ]:
# Caminho do PDF oficial da Receita Federal
CAMINHO_PDF = "P&R IRPF 2026 - v1.00 - 2026.04.23.pdf"  # ajuste o caminho se necessario

# Carrega o PDF
loader = PyPDFLoader(CAMINHO_PDF)
documents = loader.load()

# Quantidade de paginas carregadas
len(documents)

In [ ]:
documents ## O que é ?

## 2. Preparacao dos Documentos

O PDF oficial e dividido em blocos menores. Esses blocos sao usados depois para buscar os trechos mais relevantes para cada pergunta.

Chunks maiores reduzem a quantidade de chamadas de embeddings, o que ajuda a respeitar os limites do Free Tier do Gemini.

In [ ]:

# Divide os documentos em chunks menores
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

len(chunks)


In [ ]:
chunks ## mostrar 3 pedaços

## 3. Embeddings e Banco Vetorial

Cada bloco de texto e convertido em um vetor semantico. Esses vetores sao salvos em um banco local com ChromaDB.

Se o banco vetorial ja existir, ele sera carregado sem recriar os embeddings. Isso evita consumir a API novamente sem necessidade.

In [ ]:
from time import sleep

# Inicializa embeddings via Google AI Studio / Gemini
embeddings = GoogleGenerativeAIEmbeddings(
    model=GEMINI_EMBEDDING_MODEL,
    google_api_key=GOOGLE_API_KEY,
)

# Carrega o banco vetorial salvo ou cria em lotes se ainda nao existir
CHROMA_DIR = Path("./chroma_leao_claro_irpf_gemini")
vectorstore = Chroma(
    collection_name="leao_claro_irpf_gemini",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

if vectorstore._collection.count() == 0:
    batch_size = 80
    for start in range(0, len(chunks), batch_size):
        batch = chunks[start : start + batch_size]
        vectorstore.add_documents(batch)
        print(f"Indexados {min(start + batch_size, len(chunks))}/{len(chunks)} chunks")
        if start + batch_size < len(chunks):
            sleep(65)
else:
    print(f"Banco vetorial carregado com {vectorstore._collection.count()} chunks salvos")

## 4. Recuperacao de Contexto

O retriever busca os trechos do documento mais parecidos semanticamente com a pergunta do usuario.

Neste projeto, cada resposta usa os 3 trechos mais relevantes encontrados no banco vetorial.

In [ ]:

# Cria o retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


## 5. Integracao com o Modelo

O Gemini recebe dois elementos:

- a pergunta do usuario;
- os trechos oficiais recuperados pelo retriever.

O prompt limita o comportamento do assistente: ele deve responder somente com base no contexto, nao pedir dados sensiveis e nao se apresentar como substituto de contador ou da Receita Federal.

In [ ]:
# Inicializa o modelo de linguagem via Google AI Studio / Gemini
llm = ChatGoogleGenerativeAI(
    model=GEMINI_LLM_MODEL,
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Voce e o Leao Claro, um assistente informativo sobre IRPF. "
            "Responda em portugues do Brasil usando somente o contexto oficial fornecido. "
            "Nao solicite CPF, senha gov.br, dados bancarios ou informacoes sensiveis. "
            "Nao prometa economia tributaria nem substitua contador ou canais oficiais da Receita Federal. "
            "Se a resposta nao estiver no contexto, diga que nao encontrou essa informacao no documento.\n\n"
            "Contexto:\n{context}",
        ),
        ("human", "{input}"),
    ]
)

class SimpleRAGChain:
    def __init__(self, retriever, prompt, llm):
        self.retriever = retriever
        self.prompt = prompt
        self.llm = llm

    def invoke(self, inputs):
        question = inputs["input"] if isinstance(inputs, dict) else inputs
        docs = self.retriever.invoke(question)
        context = "\n\n".join(doc.page_content for doc in docs)
        messages = self.prompt.invoke({"input": question, "context": context})
        response = self.llm.invoke(messages)
        answer = response.content if hasattr(response, "content") else str(response)
        return {"input": question, "answer": answer, "context": docs}


# Cria a cadeia RAG compativel com LangChain atual
qa_chain = SimpleRAGChain(retriever, prompt, llm)

## 6. Testes e Validacao

As perguntas abaixo verificam se o assistente consegue responder usando o documento oficial e exibir os trechos que sustentam a resposta.

In [ ]:
# Pergunta de teste
pergunta = "Quais documentos devo separar antes de fazer a declaracao do IRPF?"

# Executa a pergunta no assistente RAG
resposta = qa_chain.invoke({"input": pergunta})

print("Pergunta:")
print(pergunta)

print("\nResposta do Assistente:")
print(resposta["answer"])

print("\nTrechos utilizados como contexto:\n")

for i, doc in enumerate(resposta["context"], start=1):
    print(f"--- Trecho {i} ---")
    print(f"Fonte: {doc.metadata.get('source', 'Documento desconhecido')}")
    print(f"Pagina: {doc.metadata.get('page', 'N/A')}")
    print("Conteudo:")
    print(doc.page_content)
    print("\n")

In [ ]:
# Pergunta de teste 2
pergunta = "O que e a declaracao pre-preenchida e quais cuidados devo ter ao usa-la?"

# Executa a pergunta no assistente RAG
resposta = qa_chain.invoke({"input": pergunta})

print("Pergunta:")
print(pergunta)

print("\nResposta do Assistente:")
print(resposta["answer"])

print("\nTrechos utilizados como contexto:\n")

for i, doc in enumerate(resposta["context"], start=1):
    print(f"--- Trecho {i} ---")
    print(f"Fonte: {doc.metadata.get('source', 'Documento desconhecido')}")
    print(f"Pagina: {doc.metadata.get('page', 'N/A')}")
    print("Conteudo:")
    print(doc.page_content)
    print("\n")

## Conclusao

O Leao Claro demonstra como transformar um documento oficial extenso em um assistente consultivo com RAG.

O projeto mostra:
- uso de LangChain, Gemini e ChromaDB;
- respostas baseadas em contexto recuperado;
- persistencia de embeddings para evitar custo repetido;
- cuidados de seguranca para um tema sensivel;
- uma base pronta para virar app publico no Hugging Face Spaces.

### Proximos passos
- Rodar o notebook e salvar outputs para portfolio.
- Publicar o `app.py` no Hugging Face Spaces.
- Manter `.env` fora do repositorio.
- Pedir que cada usuario informe a propria chave do Google AI Studio na demo.